# Benchmark JAX GPU Colab — PR #14e (Sprint 7.x+ — VRAM Otimizado)

**Objetivo**: reduzir consumo de VRAM (~12 GB em T4, ~60 GB em A100) via cache LRU bounded com `clear_jit_cache()` e `set_jit_cache_maxsize()`.

**Resultados reais pós-otimização Sprint 7.x (PR #14d) reportados pelo usuário:**

| Modelo | T4 (ms) | A100 (ms) | Buckets | T4 VRAM | A100 VRAM |
|:-------|--------:|----------:|--------:|--------:|----------:|
| oklahoma_3 | 23,1 | 19,0 | 5 | 12 GB | 60 GB |
| oklahoma_5 | 36,3 | 31,9 | 9 | — | — |
| oklahoma_28 | 189,3 | 160,1 | **44** | — | — |
| hou_7 | 68,9 | 62,2 | 13 | — | — |
| viking_graben_10 | 59,0 | 45,7 | 13 | — | — |

**Alvo PR #14e**: mesmo tempo, **VRAM < 4 GB em T4 e < 20 GB em A100** via `set_jit_cache_maxsize(N)`.

## 0. Setup Colab + JAX CUDA

In [ ]:
!pip install -q numpy 'numba>=0.60' 'jax[cuda12]>=0.4.30'
!git clone -b main https://github.com/daniel-guitarplayer-8/geosteering-ai.git /content/Geosteering_AI
%cd /content/Geosteering_AI

In [ ]:
import os, sys, time
os.environ['JAX_ENABLE_X64'] = 'True'
sys.path.insert(0, '/content/Geosteering_AI')

import jax
jax.config.update('jax_enable_x64', True)
print('JAX devices:', jax.devices())
print('JAX default backend:', jax.default_backend())

## 1. Cache LRU bounded — evita vazamento de VRAM

Configuração recomendada:
- **T4 (16 GB)**: `set_jit_cache_maxsize(16)` — ~1 GB/bucket × 16 ≈ 16 GB
- **A100 (40 GB)**: `set_jit_cache_maxsize(32)`
- **A100 (80 GB)**: `set_jit_cache_maxsize(64)` (default)

Entre batches de modelos, chame `clear_jit_cache()` explicitamente.

In [ ]:
import numpy as np
from geosteering_ai.simulation._jax.forward_pure import (
    build_static_context, forward_pure_jax,
    clear_jit_cache, set_jit_cache_maxsize, get_jit_cache_info,
)
from geosteering_ai.simulation.validation.canonical_models import get_canonical_model

# Ajusta conforme o hardware disponível
device_name = str(jax.devices()[0])
if 'T4' in device_name:
    set_jit_cache_maxsize(16)
    print('T4 detectada → maxsize=16 (evita OOM)')
elif 'A100' in device_name or 'A10' in device_name:
    set_jit_cache_maxsize(32)
    print('A100 detectada → maxsize=32')
else:
    set_jit_cache_maxsize(64)
    print('Default → maxsize=64')

## 2. Forward com monitoramento de cache + VRAM

In [ ]:
def bench_and_monitor(model_name: str, n_pos: int = 100, n_iter: int = 3):
    clear_jit_cache()  # limpa antes de medir
    m = get_canonical_model(model_name)
    z = np.linspace(m.min_depth - 2.0, m.max_depth + 2.0, n_pos)
    ctx = build_static_context(
        m.rho_h, m.rho_v, m.esp, z,
        np.array([20000.0]), 1.0, 0.0,
    )
    # Warmup
    t0 = time.perf_counter()
    H = forward_pure_jax(ctx.rho_h_jnp, ctx.rho_v_jnp, ctx); H.block_until_ready()
    t_first = time.perf_counter() - t0
    # Medição
    t0 = time.perf_counter()
    for _ in range(n_iter):
        H = forward_pure_jax(ctx.rho_h_jnp, ctx.rho_v_jnp, ctx); H.block_until_ready()
    t_med = (time.perf_counter() - t0) / n_iter
    info = get_jit_cache_info()

    # VRAM (se GPU disponível)
    vram_gb = None
    try:
        stats = jax.devices()[0].memory_stats()
        vram_gb = stats.get('bytes_in_use', 0) / (1024**3)
    except Exception:
        pass
    return t_first, t_med, info['n_entries'], vram_gb

print(f'{"Modelo":<20}{"1a ch (s)":>10}{"T_forward (ms)":>16}{"buckets":>10}{"VRAM (GB)":>12}')
for name in ['oklahoma_3', 'oklahoma_5', 'oklahoma_28', 'hou_7', 'viking_graben_10']:
    t_first, t_med, n_buckets, vram = bench_and_monitor(name, n_pos=100, n_iter=3)
    vram_str = f'{vram:.2f}' if vram else 'N/A'
    print(f'{name:<20}{t_first:>10.2f}{t_med*1000:>15.2f} {n_buckets:>10}{vram_str:>12}')

## 3. Pico de VRAM vs sem limite (comparação)

Demonstra que oklahoma_28 (44 buckets) com `maxsize=10` usa menos VRAM que sem limite.

In [ ]:
# Com maxsize agressivo (T4-safe)
set_jit_cache_maxsize(10)
clear_jit_cache()
m = get_canonical_model('oklahoma_28')
z = np.linspace(m.min_depth - 2, m.max_depth + 2, 100)
ctx = build_static_context(m.rho_h, m.rho_v, m.esp, z, np.array([20000.0]), 1.0, 0.0)
H = forward_pure_jax(ctx.rho_h_jnp, ctx.rho_v_jnp, ctx); H.block_until_ready()
info = get_jit_cache_info()
try:
    stats = jax.devices()[0].memory_stats()
    peak = stats.get('peak_bytes_in_use', 0) / (1024**3)
    print(f'oklahoma_28 com maxsize=10 → cache={info["n_entries"]} | peak VRAM={peak:.2f} GB')
except Exception:
    print(f'oklahoma_28 com maxsize=10 → cache={info["n_entries"]} | VRAM não disponível')

## 4. Tabela comparativa a preencher

| Modelo | maxsize | T_forward (ms) | Peak VRAM (GB) | Paridade Numba |
|:-------|--------:|--------------:|---------------:|:--------------:|
| oklahoma_3 | (ajuste) | (executar) | (executar) | < 1e-10 esperado |
| oklahoma_28 | 10 | (executar) | (executar) | < 1e-10 esperado |

Compare VRAM com o notebook `bench_jax_gpu_colab_pr14d.ipynb` — esperava-se **redução de 3-5×** em T4 para oklahoma_28.